# Journal Overlap Analysis: NLM Catalog / PMC / NIH-Funded / Northwestern Transformative Agreements

Compares four journal lists using **ISSN** as the primary match key:

| Source | Description |
|---|---|
| **NLM Catalog** | Retrieved via NCBI E-utilities |
| **PMC Journal List** | Downloaded as CSV from NCBI |
| **NIH-Funded Journals** | Top 2,228 journals with NIH-funded papers, Jan–Jul 2025 (Dimensions) |
| **Northwestern TA Journals** | Journals covered by Northwestern transformative agreements (Wiley; Springer Nature BTAA) — APCs waived |

Each journal receives:
- `in_pmc` / `in_nlm` / `in_nih` flags
- `in_northwestern_ta` — **True/False**, suitable for filtering
- `northwestern_ta_agreement` — which agreement(s) cover the journal
- NIH metadata (OA status, publisher type, APC, pub count) when matched

## 1. Imports

In [3]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import time
import pickle
import re
import io
from pathlib import Path

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ── NCBI credentials ──────────────────────────────────────────────────────────
from config import ENTREZ_EMAIL, ENTREZ_API_KEY


print('Imports OK')

Imports OK


## 2. Configuration

In [17]:
# ── NLM Catalog query ────────────────────────────────────────────────────────
# "currentlyindexed[All]"  -> MEDLINE-indexed journals      (~30k)
# "ncbijournals[All]"      -> broader NLM journal collection (~60k)
NLM_QUERY = "currentlyindexed[All]"

# ── External file paths ───────────────────────────────────────────────────────
PMC_CSV_URL  = "https://cdn.ncbi.nlm.nih.gov/pmc/home/jlist.csv"
NIH_FILE     = Path("NIH2025_2228TopJournals.csv")
WILEY_FILE   = Path("2025_08-25 Wiley Hybrid and OA Journals.csv")
SN_FILE      = Path("2025_BTAA_Springer_Nature_hybrid_journals.csv")

# ── E-utilities ───────────────────────────────────────────────────────────────
EUTILS     = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
BATCH_SIZE = 500
SLEEP_SEC  = 0.11 if ENTREZ_API_KEY else 0.34

# ── Directories ───────────────────────────────────────────────────────────────
CACHE_DIR  = Path("cache")
OUTPUT_DIR = Path("output")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

NLM_CACHE = CACHE_DIR / "nlm_journals.pkl"
PMC_CACHE = CACHE_DIR / "pmc_journals.pkl"

print(f"NLM query   : {NLM_QUERY}")
print(f"NIH file    : {NIH_FILE}   exists={NIH_FILE.exists()}")
print(f"Wiley file  : {WILEY_FILE}  exists={WILEY_FILE.exists()}")
print(f"SN file     : {SN_FILE}  exists={SN_FILE.exists()}")
print(f"API key     : {'set' if ENTREZ_API_KEY else 'not set (3 req/sec)'}")

NLM query   : currentlyindexed[All]
NIH file    : NIH2025_2228TopJournals.csv   exists=True
Wiley file  : 2025_08-25 Wiley Hybrid and OA Journals.csv  exists=True
SN file     : 2025_BTAA_Springer_Nature_hybrid_journals.csv  exists=True
API key     : set


## 3. Helper Functions

In [6]:
def normalize_issn(raw):
    """Return XXXX-XXXX string or None if not parseable."""
    if not raw or not isinstance(raw, str):
        return None
    digits = re.sub(r'[^0-9Xx]', '', raw)
    if len(digits) == 8:
        return f"{digits[:4]}-{digits[4:]}".upper()
    return None


def any_issn_in_set(row, issn_cols, target_set):
    """Return True if any normalized ISSN from the given columns hits target_set."""
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in target_set:
            return True
    return False


def entrez_params(extras=None):
    p = {"email": ENTREZ_EMAIL}
    if ENTREZ_API_KEY:
        p["api_key"] = ENTREZ_API_KEY
    if extras:
        p.update(extras)
    return p


print('Helpers defined.')

Helpers defined.


## 4. Load NIH-Funded Journal File

In [25]:
def load_nih_journals(filepath):
    for enc in ['utf-8-sig', 'cp1252', 'latin-1']:
        try:
            df = pd.read_csv(filepath, sep=None, engine='python', dtype=str, encoding=enc)
            print(f"NIH file: {len(df):,} rows | encoding: {enc} | columns: {list(df.columns)}")
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"Could not decode {filepath} with utf-8-sig, cp1252, or latin-1")
    
    
        df.columns = (
            df.columns.str.strip().str.lower()
              .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
        )

    rename_map = {
        next((c for c in df.columns if 'order' in c), None)                                      : 'nih_order',
        next((c for c in df.columns if c == 'journal'), None)                                    : 'nih_journal',
        next((c for c in df.columns if c == 'issn1'), None)                                      : 'nih_issn1',
        next((c for c in df.columns if c == 'issn2'), None)                                      : 'nih_issn2',
        next((c for c in df.columns if c == 'issn3'), None)                                      : 'nih_issn3',
        next((c for c in df.columns if c == 'issn4'), None)                                      : 'nih_issn4',
        next((c for c in df.columns if 'publisher' in c and 'type' not in c), None)              : 'nih_publisher',
        next((c for c in df.columns if 'publications' in c and 'nih' in c), None)                : 'nih_pub_count',
        next((c for c in df.columns if 'oa_status' in c or ('oa' in c and 'status' in c)), None) : 'nih_oa_status',
        next((c for c in df.columns if 'publisher_type' in c), None)                             : 'nih_publisher_type',
        next((c for c in df.columns if 'apc_2025' in c or ('apc' in c and '2025' in c)), None)  : 'nih_apc_2025_usd',
        next((c for c in df.columns if 'apc_category' in c), None)                              : 'nih_apc_category',
    }
    rename_map = {k: v for k, v in rename_map.items() if k is not None}
    df = df.rename(columns=rename_map)

    for i in range(1, 5):
        matches = [c for c in df.columns if c.strip().upper() == f'ISSN{i}']
        if matches:
            df[f'nih_issn{i}'] = df[matches[0]].apply(normalize_issn)
        else:
            df[f'nih_issn{i}'] = None

    for col in ['nih_pub_count', 'nih_apc_2025_usd', 'nih_order']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


def build_nih_lookup(nih_df):
    keep = [c for c in [
        'nih_order', 'nih_journal', 'nih_publisher',
        'nih_pub_count', 'nih_oa_status', 'nih_publisher_type',
        'nih_apc_2025_usd', 'nih_apc_category',
    ] if c in nih_df.columns]
    lookup = {}
    for _, row in nih_df.iterrows():
        entry = {c: row.get(c) for c in keep}
        for col in ['nih_issn1', 'nih_issn2', 'nih_issn3', 'nih_issn4']:
            v = row.get(col)
            if pd.notna(v) and v:
                lookup[v] = entry
    return lookup


nih_df     = load_nih_journals(NIH_FILE)
nih_lookup = build_nih_lookup(nih_df)
nih_issns  = set(nih_lookup.keys())
print(f"Unique ISSNs in NIH file: {len(nih_issns):,}")
nih_df.head(2)

NIH file: 2,228 rows | encoding: cp1252 | columns: ['Order (# NIH papers 01-07/2025)', 'Journal', 'ISSN1', 'ISSN2', 'ISSN3', 'ISSN4', 'Publisher', 'Publications acknowledging NIH funding (01-07/2025)', 'Journal OA status', 'Publisher type', 'APC 2025 (USD)', 'APC category', 'Total APCs (based on 01-07/2025)', 'APCs covered by $2k cap', '% covered by $2k cap', 'APCs not covered by $2k cap', 'APCs covered by $3k cap', '% covered by $3k cap', 'APCs not covered by $3k cap', 'APCs covered by $6 cap', '% covered by $6 cap', 'APCs not covered by $6 cap']
Unique ISSNs in NIH file: 3,860


,Order (# NIH papers 01-07/2025),Journal,ISSN1,ISSN2,ISSN3,ISSN4,Publisher,Publications acknowledging NIH funding (01-07/2025),Journal OA status,Publisher type,...,APCs covered by $3k cap,% covered by $3k cap,APCs not covered by $3k cap,APCs covered by $6 cap,% covered by $6 cap,APCs not covered by $6 cap,nih_issn1,nih_issn2,nih_issn3,nih_issn4
0,1,Nature Communications,2041-1723,NaN,NaN,NaN,Springer Nature,1300,gold,for profit,...,3900000,0.429184549,5187000,7800000,0.858369099,1287000,2041-1723,None,None,None
1,2,Scientific Reports,2045-2322,NaN,NaN,NaN,Springer Nature,812,gold,for profit,...,2184280,1,0,2184280,1,0,2045-2322,None,None,None


## 5. Load Northwestern Transformative Agreement Files

**Wiley** (`2025_08-25 Wiley Hybrid and OA Journals.xlsx`): `Journal Title`, `Online ISSN`, `Type`  
**Springer Nature BTAA** (`2025_BTAA_Springer_Nature_hybrid_journals.xlsx`): `Journal Title`, `eISSN`, `Publishing Model`, `OA License`

Each journal gets:
- `in_northwestern_ta` — **True/False** filter column
- `northwestern_ta_agreement` — "Wiley", "Springer Nature", or "Wiley; Springer Nature"
- `ta_publishing_model` — journal type/model from the agreement file

In [26]:
def load_wiley_ta(filepath):
    """
    Load Wiley TA file.
    Expected columns: Journal Title, Online ISSN, Type
    Returns DataFrame with normalized columns and ISSN.
    """
    for enc in ['utf-8-sig', 'cp1252', 'latin-1']:
        try:
            df = pd.read_csv(filepath, sep=None, engine='python', dtype=str, encoding=enc)
            print(f"NIH file: {len(df):,} rows | encoding: {enc} | columns: {list(df.columns)}")
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"Could not decode {filepath} with utf-8-sig, cp1252, or latin-1")
        print(f"Wiley file: {len(df):,} rows | columns: {list(df.columns)}")

    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    # Map column variants
    rename_map = {}
    for c in df.columns:
        if re.search(r'title|journal', c):    rename_map[c] = 'ta_title'
        elif re.search(r'issn', c):           rename_map[c] = 'ta_issn_raw'
        elif re.search(r'type|model', c):     rename_map[c] = 'ta_publishing_model'
    df = df.rename(columns=rename_map)

    if 'ta_issn_raw' not in df.columns:
        raise ValueError("Could not find ISSN column in Wiley file. Check column headers.")

    df['ta_issn'] = df['ta_issn_raw'].apply(normalize_issn)
    df['ta_agreement'] = 'Wiley'

    before = len(df)
    df = df[df['ta_issn'].notna()].copy()
    print(f"  Wiley rows with valid ISSN: {len(df):,} (dropped {before - len(df):,} no-ISSN rows)")
    return df[['ta_title', 'ta_issn', 'ta_publishing_model', 'ta_agreement']]


def load_sn_ta(filepath):
    """
    Load Springer Nature BTAA TA file.
    Expected columns: Journal Title, eISSN, Publishing Model, OA License
    Returns DataFrame with normalized columns and ISSN.
    """
    for enc in ['utf-8-sig', 'cp1252', 'latin-1']:
        try:
            df = pd.read_csv(filepath, sep=None, engine='python', dtype=str, encoding=enc)
            print(f"Springer Nature file: {len(df):,} rows | encoding: {enc} | columns: {list(df.columns)}")
            break
        except UnicodeDecodeError:
            continue


    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    rename_map = {}
    for c in df.columns:
        if re.search(r'title', c) and 'imprint' not in c: rename_map[c] = 'ta_title'
        elif re.search(r'^e_?issn$|eissn', c):                    rename_map[c] = 'ta_issn_raw'
        elif re.search(r'publishing_model|model', c):              rename_map[c] = 'ta_publishing_model'
        elif re.search(r'oa_license|license', c):                  rename_map[c] = 'ta_oa_license'
    df = df.rename(columns=rename_map)

    if 'ta_issn_raw' not in df.columns:
        raise ValueError("Could not find eISSN column in Springer Nature file. Check column headers.")

    df['ta_issn'] = df['ta_issn_raw'].apply(normalize_issn)
    df['ta_agreement'] = 'Springer Nature'

    before = len(df)
    df = df[df['ta_issn'].notna()].copy()
    print(f"  Springer Nature rows with valid ISSN: {len(df):,} (dropped {before - len(df):,} no-ISSN rows)")

    keep = ['ta_title', 'ta_issn', 'ta_publishing_model', 'ta_agreement']
    if 'ta_oa_license' in df.columns:
        keep.append('ta_oa_license')
    return df[keep]


wiley_df = load_wiley_ta(WILEY_FILE)
sn_df    = load_sn_ta(SN_FILE)

# Combine both TA lists
ta_df = pd.concat([wiley_df, sn_df], ignore_index=True)

# Build per-ISSN lookup: issn -> {'ta_agreement': ..., 'ta_publishing_model': ...}
# If a journal appears in both, record both agreement names
ta_lookup = {}   # issn -> dict
for _, row in ta_df.iterrows():
    issn = row['ta_issn']
    if not issn:
        continue
    model  = row.get('ta_publishing_model', '')
    oa_lic = row.get('ta_oa_license', '')
    agmt   = row['ta_agreement']
    if issn in ta_lookup:
        # Journal in both agreements — merge agreement names
        existing = ta_lookup[issn]
        if agmt not in existing['ta_agreement']:
            existing['ta_agreement'] += f"; {agmt}"
    else:
        ta_lookup[issn] = {
            'ta_agreement'       : agmt,
            'ta_publishing_model': model,
            'ta_oa_license'      : oa_lic,
        }

ta_issns = set(ta_lookup.keys())

print(f"\nWiley TA journals       : {len(wiley_df):,}")
print(f"Springer Nature TA journals: {len(sn_df):,}")
print(f"Combined unique TA ISSNs: {len(ta_issns):,}")
wiley_df.head(2)

NIH file: 1,847 rows | encoding: utf-8-sig | columns: ['Journal Title', 'Online ISSN', 'Type']
  Wiley rows with valid ISSN: 1,847 (dropped 0 no-ISSN rows)
Springer Nature file: 2,041 rows | encoding: cp1252 | columns: ['S/N', 'Journal ID', 'Journal Title', 'eISSN', 'Journal Imprint', 'Main Discipline', 'Publishing Model', 'OA License', 'URL']
  Springer Nature rows with valid ISSN: 2,041 (dropped 0 no-ISSN rows)

Wiley TA journals       : 1,847
Springer Nature TA journals: 2,041
Combined unique TA ISSNs: 3,888


,ta_title,ta_issn,ta_publishing_model,ta_agreement
0,Abacus,1467-6281,Hybrid,Wiley
1,Academic Emergency Medicine,1553-2712,Hybrid,Wiley


## 6. Fetch PMC Journal List

In [27]:
def fetch_pmc_journals(use_cache=True):
    if use_cache and PMC_CACHE.exists():
        print(f"Loading PMC from cache: {PMC_CACHE}")
        with open(PMC_CACHE, 'rb') as f:
            return pickle.load(f)

    print(f"Downloading PMC journal list...")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
    resp = requests.get(PMC_CSV_URL, timeout=60)
    
    resp.raise_for_status()

    try:
        df = pd.read_csv(io.StringIO(resp.content.decode('utf-8-sig')))
    except Exception:
        df = pd.read_csv(io.StringIO(resp.content.decode('latin-1')))

    print(f"Raw PMC columns: {list(df.columns)}")

    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    col_map = {}
    for c in df.columns:
        if re.search(r'^journal_title$|^journal_name$|journal.*name', c):          col_map[c] = 'pmc_title'
        elif re.search(r'^e_?issn$|electronic.*issn|issn.*online|online.*issn', c): col_map[c] = 'pmc_eissn'
        elif re.search(r'^issn$|print.*issn|p_?issn|issn.*print', c):              col_map[c] = 'pmc_issn'
        elif re.search(r'pmc.*id|journal.*id', c):                                  col_map[c] = 'pmc_id'
        elif re.search(r'particip|agreement_status', c):                            col_map[c] = 'pmc_participation'
    df = df.rename(columns=col_map)

    for col in ['pmc_title', 'pmc_issn', 'pmc_eissn']:
        if col not in df.columns:
            df[col] = np.nan

    df['pmc_issn_norm']  = df['pmc_issn'].apply(normalize_issn)
    df['pmc_eissn_norm'] = df['pmc_eissn'].apply(normalize_issn)

    before = len(df)
    df = df[df['pmc_issn_norm'].notna() | df['pmc_eissn_norm'].notna()].copy()
    print(f"PMC: {len(df):,} rows with ISSN (dropped {before - len(df):,})")

    with open(PMC_CACHE, 'wb') as f:
        pickle.dump(df, f)
    return df


pmc_df = fetch_pmc_journals(use_cache=False)
print(f"PMC journals loaded: {len(pmc_df):,}")
pmc_df.head()

Raw PMC columns: ['Journal Title', 'NLM Title Abbreviation (TA)', 'Publisher', 'ISSN (print)', 'ISSN (online)', 'NLM Unique ID', 'Most Recent', 'Earliest', 'Release Delay (Embargo)', 'Agreement Status', 'Agreement to Deposit', 'Journal Note', 'PMC URL']
PMC: 4,372 rows with ISSN (dropped 79)
PMC journals loaded: 4,372


,pmc_title,nlm_title_abbreviation_ta,publisher,pmc_issn,pmc_eissn,nlm_unique_id,most_recent,earliest,release_delay_embargo,pmc_participation,agreement_to_deposit,journal_note,pmc_url,pmc_issn_norm,pmc_eissn_norm
0,3 Biotech,3 Biotech,Springer,2190-572X,2190-5738,101565857,v.16(4) 2026,v.1(1) 2011,12 months,Active,All articles,NaN,"https://www.ncbi.nlm.nih.gov/pmc/?term=""3 Biot...",2190-572X,2190-5738
1,3D Printing and Additive Manufacturing,3D Print Addit Manuf,"Mary Ann Liebert, Inc.",2329-7662,2329-7670,101649453,v.12(3) 2025,v.7(1) 2020,12 months,Active,All articles,NaN,"https://www.ncbi.nlm.nih.gov/pmc/?term=""3D Pri...",2329-7662,2329-7670
2,3D Printing in Medicine,3D Print Med,BMC,NaN,2365-6271,101721758,v.12 2026,v.2 2016,0 months (Immediate release),Active,All articles,NaN,"https://www.ncbi.nlm.nih.gov/pmc/?term=""3D Pri...",None,2365-6271
3,AACE Clinical Case Reports,AACE Clin Case Rep,American Association of Clinical Endocrinology,NaN,2376-0605,101670593,v.11(2) 2025,v.3(2) 2017,0 months (Immediate release),Predecessor title,All articles,NaN,"https://www.ncbi.nlm.nih.gov/pmc/?term=""AACE C...",None,2376-0605
4,AACE Endocrinology and Diabetes,AACE Endocrinol Diabetes,American Association of Clinical Endocrinology,NaN,3050-9157,9919052034906676,v.13(1) 2026,v.12(1) 2025,0 months (Immediate release),Active,All articles,NaN,"https://www.ncbi.nlm.nih.gov/pmc/?term=""AACE E...",None,3050-9157


## 7. Fetch NLM Catalog Journals via E-utilities

In [28]:
def esearch_nlm(query):
    params = entrez_params({"db": "nlmcatalog", "term": query, "usehistory": "y", "retmax": 0})
    resp = requests.get(f"{EUTILS}/esearch.fcgi", params=params, timeout=60)
    resp.raise_for_status()
    root = ET.fromstring(resp.text)
    return root.findtext('WebEnv'), root.findtext('QueryKey'), int(root.findtext('Count', '0'))


def efetch_batch(webenv, query_key, start, retmax):
    params = entrez_params({
        "db": "nlmcatalog", "query_key": query_key, "WebEnv": webenv,
        "retstart": start, "retmax": retmax, "rettype": "xml", "retmode": "xml",
    })
    resp = requests.get(f"{EUTILS}/efetch.fcgi", params=params, timeout=120)
    resp.raise_for_status()
    return resp.text


def parse_nlm_xml(xml_text):
    records = []
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        print("  WARNING: XML parse error in batch — skipping")
        return records
    for rec in root.findall('.//NLMCatalogRecord'):
        nlm_id     = rec.findtext('NlmUniqueID', '')
        title_el   = rec.find('.//TitleMain/Title')
        title      = title_el.text.strip() if title_el is not None and title_el.text else ''
        medline_ta = rec.findtext('MedlineTA', '')
        linking    = normalize_issn(rec.findtext('ISSNLinking'))
        print_issn = e_issn = None
        for issn_el in rec.findall('.//ISSN'):
            issn_type = issn_el.get('IssnType', '').lower()
            val = normalize_issn(issn_el.text)
            if issn_type == 'print' and val:      print_issn = val
            elif issn_type == 'electronic' and val: e_issn = val
        records.append({
            'nlm_id': nlm_id, 'nlm_title': title, 'medline_ta': medline_ta,
            'nlm_issn': print_issn, 'nlm_eissn': e_issn, 'nlm_linking': linking,
        })
    return records


def fetch_nlm_journals(query, use_cache=True):
    if use_cache and NLM_CACHE.exists():
        print(f"Loading NLM from cache: {NLM_CACHE}")
        with open(NLM_CACHE, 'rb') as f:
            return pickle.load(f)

    print(f"Searching NLM Catalog: '{query}'")
    webenv, query_key, count = esearch_nlm(query)
    print(f"Total records: {count:,}")

    all_records = []
    for start in tqdm(range(0, count, BATCH_SIZE), desc="Fetching NLM batches"):
        retmax = min(BATCH_SIZE, count - start)
        for attempt in range(3):
            try:
                all_records.extend(parse_nlm_xml(efetch_batch(webenv, query_key, start, retmax)))
                break
            except requests.HTTPError as e:
                print(f"  HTTP error at start={start}, attempt {attempt+1}: {e}")
                time.sleep(2 ** attempt)
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(all_records)
    print(f"NLM records parsed: {len(df):,}")
    with open(NLM_CACHE, 'wb') as f:
        pickle.dump(df, f)
    return df


nlm_df = fetch_nlm_journals(NLM_QUERY, use_cache=True)
print(f"NLM Catalog journals loaded: {len(nlm_df):,}")
nlm_df.head(2)

Loading NLM from cache: cache\nlm_journals.pkl
NLM Catalog journals loaded: 5,227


,nlm_id,nlm_title,medline_ta,nlm_issn,nlm_eissn,nlm_linking
0,9919253715206676,MedScience.,MedScience,None,3091-4981,3091-4973
1,9919227857106676,Circulation. Population health and outcomes.,Circ Popul Health Outcomes,None,3068-563X,3068-563X


## 8. Build ISSN Lookup Sets

In [29]:
pmc_issns = set()
for _, row in pmc_df.iterrows():
    for v in [row.get('pmc_issn_norm'), row.get('pmc_eissn_norm')]:
        if pd.notna(v) and v: pmc_issns.add(v)

nlm_issns = set()
for _, row in nlm_df.iterrows():
    for col in ['nlm_issn', 'nlm_eissn', 'nlm_linking']:
        v = row.get(col)
        if pd.notna(v) and v: nlm_issns.add(v)

# nih_issns and ta_issns built in Sections 4 and 5

print(f"Unique ISSNs  NLM : {len(nlm_issns):,}")
print(f"Unique ISSNs  PMC : {len(pmc_issns):,}")
print(f"Unique ISSNs  NIH : {len(nih_issns):,}")
print(f"Unique ISSNs  TA  : {len(ta_issns):,}")
print()
print(f"NLM n PMC      : {len(nlm_issns & pmc_issns):,}")
print(f"NLM n NIH      : {len(nlm_issns & nih_issns):,}")
print(f"NLM n TA       : {len(nlm_issns & ta_issns):,}")
print(f"PMC n NIH      : {len(pmc_issns & nih_issns):,}")
print(f"PMC n TA       : {len(pmc_issns & ta_issns):,}")
print(f"NIH n TA       : {len(nih_issns & ta_issns):,}")
print(f"All four       : {len(nlm_issns & pmc_issns & nih_issns & ta_issns):,}")

Unique ISSNs  NLM : 9,726
Unique ISSNs  PMC : 6,756
Unique ISSNs  NIH : 3,860
Unique ISSNs  TA  : 3,888

NLM n PMC      : 2,415
NLM n NIH      : 2,967
NLM n TA       : 1,097
PMC n NIH      : 1,331
PMC n TA       : 375
NIH n TA       : 452
All four       : 68


NIH df shape: (2228, 26)
NIH df columns: ['Order (# NIH papers 01-07/2025)', 'Journal', 'ISSN1', 'ISSN2', 'ISSN3', 'ISSN4', 'Publisher', 'Publications acknowledging NIH funding (01-07/2025)', 'Journal OA status', 'Publisher type', 'APC 2025 (USD)', 'APC category', 'Total APCs (based on 01-07/2025)', 'APCs covered by $2k cap', '% covered by $2k cap', 'APCs not covered by $2k cap', 'APCs covered by $3k cap', '% covered by $3k cap', 'APCs not covered by $3k cap', 'APCs covered by $6 cap', '% covered by $6 cap', 'APCs not covered by $6 cap', 'nih_issn1', 'nih_issn2', 'nih_issn3', 'nih_issn4']

nih_issn1 — sample values: []
nih_issn2 — sample values: []
nih_issn3 — sample values: []
nih_issn4 — sample values: []


## 9. Classify NLM Catalog Journals

In [30]:
NLM_ISSN_COLS = ['nlm_issn', 'nlm_eissn', 'nlm_linking']

nlm_df['in_pmc'] = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, pmc_issns), axis=1)
nlm_df['in_nih'] = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, nih_issns), axis=1)
nlm_df['in_northwestern_ta'] = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, ta_issns), axis=1)

print("NLM Catalog classification:")
print(f"  in PMC                 : {nlm_df['in_pmc'].sum():,}")
print(f"  in NIH list            : {nlm_df['in_nih'].sum():,}")
print(f"  in Northwestern TA     : {nlm_df['in_northwestern_ta'].sum():,}")
print(f"  in NIH + TA (both)     : {(nlm_df['in_nih'] & nlm_df['in_northwestern_ta']).sum():,}")

NLM Catalog classification:
  in PMC                 : 1,395
  in NIH list            : 1,616
  in Northwestern TA     : 1,097
  in NIH + TA (both)     : 377


## 10. Classify PMC Journals

In [31]:
PMC_ISSN_COLS = ['pmc_issn_norm', 'pmc_eissn_norm']

pmc_df['in_nlm'] = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nlm_issns), axis=1)
pmc_df['in_nih'] = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nih_issns), axis=1)
pmc_df['in_northwestern_ta'] = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, ta_issns), axis=1)

print("PMC classification:")
print(f"  in NLM Catalog         : {pmc_df['in_nlm'].sum():,}")
print(f"  in NIH list            : {pmc_df['in_nih'].sum():,}")
print(f"  in Northwestern TA     : {pmc_df['in_northwestern_ta'].sum():,}")
print(f"  in NIH + TA (both)     : {(pmc_df['in_nih'] & pmc_df['in_northwestern_ta']).sum():,}")

PMC classification:
  in NLM Catalog         : 1,398
  in NIH list            : 948
  in Northwestern TA     : 375
  in NIH + TA (both)     : 100


## 11. Join NIH Metadata and TA Metadata

In [32]:
NIH_META_COLS = [
    'nih_order', 'nih_journal', 'nih_publisher',
    'nih_pub_count', 'nih_oa_status', 'nih_publisher_type',
    'nih_apc_2025_usd', 'nih_apc_category',
]
NIH_EMPTY = {c: None for c in NIH_META_COLS}

TA_META_COLS = ['northwestern_ta_agreement', 'ta_publishing_model', 'ta_oa_license']
TA_EMPTY     = {c: None for c in TA_META_COLS}


def get_nih_meta(row, issn_cols):
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in nih_lookup:
            return nih_lookup[v]
    return NIH_EMPTY.copy()


def get_ta_meta(row, issn_cols):
    """Return TA metadata dict, or empty dict if no ISSN match."""
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in ta_lookup:
            return ta_lookup[v]
    return TA_EMPTY.copy()


# Join to NLM frame
nlm_df = pd.concat([
    nlm_df.reset_index(drop=True),
    pd.DataFrame(nlm_df.apply(lambda r: get_nih_meta(r, NLM_ISSN_COLS), axis=1).tolist()),
    pd.DataFrame(nlm_df.apply(lambda r: get_ta_meta(r, NLM_ISSN_COLS), axis=1).tolist()),
], axis=1)

# Join to PMC frame
pmc_df = pd.concat([
    pmc_df.reset_index(drop=True),
    pd.DataFrame(pmc_df.apply(lambda r: get_nih_meta(r, PMC_ISSN_COLS), axis=1).tolist()),
    pd.DataFrame(pmc_df.apply(lambda r: get_ta_meta(r, PMC_ISSN_COLS), axis=1).tolist()),
], axis=1)

print("Metadata joined.")
print(f"NLM frame shape: {nlm_df.shape}")
print(f"PMC frame shape: {pmc_df.shape}")

Metadata joined.
NLM frame shape: (5227, 21)
PMC frame shape: (4372, 30)


## 12. Build Comparison Tables

In [33]:
NLM_BASE = [
    'nlm_id', 'nlm_title', 'medline_ta', 'nlm_issn', 'nlm_eissn', 'nlm_linking',
    'in_pmc', 'in_nih', 'in_northwestern_ta',
]
PMC_BASE = ['pmc_title', 'pmc_issn_norm', 'pmc_eissn_norm', 'in_nlm', 'in_nih', 'in_northwestern_ta']
for opt in ['pmc_participation', 'pmc_id']:
    if opt in pmc_df.columns:
        PMC_BASE.append(opt)

NIH_COLS_PRESENT = [c for c in NIH_META_COLS if c in nlm_df.columns]
TA_COLS_PRESENT  = [c for c in TA_META_COLS  if c in nlm_df.columns]
META_COLS = NIH_COLS_PRESENT + TA_COLS_PRESENT


def make_nlm_table(mask, label):
    cols = [c for c in NLM_BASE + META_COLS if c in nlm_df.columns]
    df = nlm_df.loc[mask, cols].copy().reset_index(drop=True)
    df.insert(0, 'category', label)
    return df


def make_pmc_table(mask, label):
    pmc_meta = [c for c in META_COLS if c in pmc_df.columns]
    cols = [c for c in PMC_BASE + pmc_meta if c in pmc_df.columns]
    df = pmc_df.loc[mask, cols].copy().reset_index(drop=True)
    df.insert(0, 'category', label)
    return df


# NLM-based slices
df_nlm_pmc_nih   = make_nlm_table(nlm_df['in_pmc'] & nlm_df['in_nih'],   'NLM + PMC + NIH')
df_nlm_pmc       = make_nlm_table(nlm_df['in_pmc'] & ~nlm_df['in_nih'],  'NLM + PMC (not NIH)')
df_nlm_nih       = make_nlm_table(~nlm_df['in_pmc'] & nlm_df['in_nih'],  'NLM + NIH (not PMC)')
df_nlm_only      = make_nlm_table(~nlm_df['in_pmc'] & ~nlm_df['in_nih'], 'NLM Only')

# PMC journals not in NLM
df_pmc_nih       = make_pmc_table(~pmc_df['in_nlm'] & pmc_df['in_nih'],  'PMC + NIH (not NLM)')
df_pmc_only      = make_pmc_table(~pmc_df['in_nlm'] & ~pmc_df['in_nih'], 'PMC Only')

# NIH journals not in NLM or PMC
nih_df['in_nlm'] = nih_df.apply(
    lambda r: any_issn_in_set(r, ['nih_issn1','nih_issn2','nih_issn3','nih_issn4'], nlm_issns), axis=1
)
nih_df['in_pmc'] = nih_df.apply(
    lambda r: any_issn_in_set(r, ['nih_issn1','nih_issn2','nih_issn3','nih_issn4'], pmc_issns), axis=1
)
nih_df['in_northwestern_ta'] = nih_df.apply(
    lambda r: any_issn_in_set(r, ['nih_issn1','nih_issn2','nih_issn3','nih_issn4'], ta_issns), axis=1
)
df_nih_only = nih_df[~nih_df['in_nlm'] & ~nih_df['in_pmc']].copy().reset_index(drop=True)
df_nih_only.insert(0, 'category', 'NIH Only')

# Northwestern TA journals — cross-cutting view
df_ta_nlm = make_nlm_table(nlm_df['in_northwestern_ta'], 'Northwestern TA (NLM match)')
df_ta_pmc = make_pmc_table(pmc_df['in_northwestern_ta'] & ~pmc_df['in_nlm'], 'Northwestern TA (PMC only)')

print("Comparison table row counts:")
for label, df in [
    ('NLM + PMC + NIH',                df_nlm_pmc_nih),
    ('NLM + PMC (not NIH)',            df_nlm_pmc),
    ('NLM + NIH (not PMC)',            df_nlm_nih),
    ('NLM Only',                       df_nlm_only),
    ('PMC + NIH (not NLM)',            df_pmc_nih),
    ('PMC Only',                       df_pmc_only),
    ('NIH Only',                       df_nih_only),
    ('Northwestern TA (NLM match)',    df_ta_nlm),
    ('Northwestern TA (PMC only)',     df_ta_pmc),
]:
    print(f"  {label:<33}: {len(df):>5,}")

Comparison table row counts:
  NLM + PMC + NIH                  :   541
  NLM + PMC (not NIH)              :   854
  NLM + NIH (not PMC)              : 1,075
  NLM Only                         : 2,757
  PMC + NIH (not NLM)              :   407
  PMC Only                         : 2,567
  NIH Only                         :   232
  Northwestern TA (NLM match)      : 1,097
  Northwestern TA (PMC only)       :   220


## 13. Summary Statistics

In [34]:
n_nlm = len(nlm_df)
n_pmc = len(pmc_df)
n_nih = len(nih_df)
n_ta  = len(ta_issns)

def pct(n, d):
    return f"{n/d*100:.1f}%" if d > 0 else '—'

summary_rows = [
    {'Category': 'In NLM + PMC + NIH',
     'Count': len(df_nlm_pmc_nih),
     '% of NLM': pct(len(df_nlm_pmc_nih), n_nlm),
     '% of PMC': pct(len(df_nlm_pmc_nih), n_pmc),
     '% of NIH': pct(len(df_nlm_pmc_nih), n_nih)},
    {'Category': 'In NLM + PMC (not NIH)',
     'Count': len(df_nlm_pmc),
     '% of NLM': pct(len(df_nlm_pmc), n_nlm),
     '% of PMC': pct(len(df_nlm_pmc), n_pmc), '% of NIH': '—'},
    {'Category': 'In NLM + NIH (not PMC)',
     'Count': len(df_nlm_nih),
     '% of NLM': pct(len(df_nlm_nih), n_nlm),
     '% of PMC': '—',
     '% of NIH': pct(len(df_nlm_nih), n_nih)},
    {'Category': 'In NLM Only',
     'Count': len(df_nlm_only),
     '% of NLM': pct(len(df_nlm_only), n_nlm), '% of PMC': '—', '% of NIH': '—'},
    {'Category': 'In PMC + NIH (not NLM)',
     'Count': len(df_pmc_nih),
     '% of NLM': '—',
     '% of PMC': pct(len(df_pmc_nih), n_pmc),
     '% of NIH': pct(len(df_pmc_nih), n_nih)},
    {'Category': 'In PMC Only',
     'Count': len(df_pmc_only),
     '% of NLM': '—',
     '% of PMC': pct(len(df_pmc_only), n_pmc), '% of NIH': '—'},
    {'Category': 'In NIH Only',
     'Count': len(df_nih_only),
     '% of NLM': '—', '% of PMC': '—',
     '% of NIH': pct(len(df_nih_only), n_nih)},
    {'Category': '─' * 32, 'Count': '', '% of NLM': '', '% of PMC': '', '% of NIH': ''},
    {'Category': 'Northwestern TA — matched in NLM',
     'Count': nlm_df['in_northwestern_ta'].sum(),
     '% of NLM': pct(int(nlm_df['in_northwestern_ta'].sum()), n_nlm),
     '% of PMC': '—', '% of NIH': '—'},
    {'Category': 'Northwestern TA — matched in PMC',
     'Count': pmc_df['in_northwestern_ta'].sum(),
     '% of NLM': '—',
     '% of PMC': pct(int(pmc_df['in_northwestern_ta'].sum()), n_pmc),
     '% of NIH': '—'},
    {'Category': 'Northwestern TA — matched in NIH list',
     'Count': nih_df['in_northwestern_ta'].sum(),
     '% of NLM': '—', '% of PMC': '—',
     '% of NIH': pct(int(nih_df['in_northwestern_ta'].sum()), n_nih)},
    {'Category': '─' * 32, 'Count': '', '% of NLM': '', '% of PMC': '', '% of NIH': ''},
    {'Category': 'Total NLM Catalog journals',
     'Count': n_nlm, '% of NLM': '100.0%', '% of PMC': '—', '% of NIH': '—'},
    {'Category': 'Total PMC journals',
     'Count': n_pmc, '% of NLM': '—', '% of PMC': '100.0%', '% of NIH': '—'},
    {'Category': 'Total NIH-funded journals',
     'Count': n_nih, '% of NLM': '—', '% of PMC': '—', '% of NIH': '100.0%'},
    {'Category': 'Total Northwestern TA journals (unique ISSNs)',
     'Count': n_ta, '% of NLM': '—', '% of PMC': '—', '% of NIH': '—'},
]

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
summary

                                     Category Count % of NLM % of PMC % of NIH
                           In NLM + PMC + NIH   541    10.4%    12.4%    24.3%
                       In NLM + PMC (not NIH)   854    16.3%    19.5%        —
                       In NLM + NIH (not PMC)  1075    20.6%        —    48.2%
                                  In NLM Only  2757    52.7%        —        —
                       In PMC + NIH (not NLM)   407        —     9.3%    18.3%
                                  In PMC Only  2567        —    58.7%        —
                                  In NIH Only   232        —        —    10.4%
             ────────────────────────────────                                 
             Northwestern TA — matched in NLM  1097    21.0%        —        —
             Northwestern TA — matched in PMC   375        —     8.6%        —
        Northwestern TA — matched in NIH list   452        —        —    20.3%
             ────────────────────────────────       

,Category,Count,% of NLM,% of PMC,% of NIH
0,In NLM + PMC + NIH,541,10.4%,12.4%,24.3%
1,In NLM + PMC (not NIH),854,16.3%,19.5%,—
2,In NLM + NIH (not PMC),1075,20.6%,—,48.2%
3,In NLM Only,2757,52.7%,—,—
4,In PMC + NIH (not NLM),407,—,9.3%,18.3%
5,In PMC Only,2567,—,58.7%,—
6,In NIH Only,232,—,—,10.4%
7,────────────────────────────────,,,,
8,Northwestern TA — matched in NLM,1097,21.0%,—,—
9,Northwestern TA — matched in PMC,375,—,8.6%,—


## 14. Quick Preview — Northwestern TA Journals with NIH Activity

In [35]:
# NLM journals that are in the TA AND in the NIH list — sorted by NIH rank
ta_nih_mask = nlm_df['in_northwestern_ta'] & nlm_df['in_nih']
preview_cols = ['nlm_title', 'nlm_issn', 'nlm_eissn',
                'northwestern_ta_agreement', 'ta_publishing_model',
                'nih_order', 'nih_pub_count', 'nih_oa_status',
                'nih_apc_2025_usd', 'nih_apc_category']
preview_cols = [c for c in preview_cols if c in nlm_df.columns]

ta_nih_df = nlm_df.loc[ta_nih_mask, preview_cols].sort_values('nih_order')
print(f"Northwestern TA journals also on NIH-funded list: {len(ta_nih_df):,}")
ta_nih_df.head(15)

Northwestern TA journals also on NIH-funded list: 377


,nlm_title,nlm_issn,nlm_eissn,northwestern_ta_agreement,ta_publishing_model,nih_order,nih_pub_count,nih_oa_status,nih_apc_2025_usd,nih_apc_category
10,Journal of imaging informatics in medicine.,2948-2925,2948-2933,NaN,Springer Hybrid,NaN,NaN,NaN,NaN,NaN
22,"Alcohol, clinical & experimental research.",None,2993-7175,NaN,Hybrid,NaN,NaN,NaN,NaN,NaN
24,American journal of biological anthropology.,None,2692-7691,NaN,Hybrid,NaN,NaN,NaN,NaN,NaN
69,Advanced biology.,None,2701-0198,NaN,Hybrid,NaN,NaN,NaN,NaN,NaN
76,Current protocols.,None,2691-1299,NaN,Hybrid,NaN,NaN,NaN,NaN,NaN
92,Research on child and adolescent psychopathology.,2730-7166,2730-7174,NaN,Springer Hybrid,NaN,NaN,NaN,NaN,NaN
191,"Endocrinology, diabetes & metabolism.",None,2398-9238,NaN,Fully OA,NaN,NaN,NaN,NaN,NaN
203,Small methods.,None,2366-9608,NaN,Hybrid,NaN,NaN,NaN,NaN,NaN
226,Birth defects research.,None,2472-1727,NaN,Hybrid,NaN,NaN,NaN,NaN,NaN
237,GeroScience.,2509-2715,2509-2723,NaN,Springer Hybrid,NaN,NaN,NaN,NaN,NaN


## 15. Export to Excel

In [36]:
output_path = OUTPUT_DIR / "nlm_pmc_nih_ta_journal_comparison.xlsx"

def drop_cat(df):
    return df.drop(columns=['category'], errors='ignore')

sheets = [
    ('Summary',            summary),
    ('NLM+PMC+NIH',        drop_cat(df_nlm_pmc_nih)),
    ('NLM+PMC_notNIH',     drop_cat(df_nlm_pmc)),
    ('NLM+NIH_notPMC',     drop_cat(df_nlm_nih)),
    ('NLM_Only',           drop_cat(df_nlm_only)),
    ('PMC+NIH_notNLM',     drop_cat(df_pmc_nih)),
    ('PMC_Only',           drop_cat(df_pmc_only)),
    ('NIH_Only',           drop_cat(df_nih_only)),
    # Northwestern TA cross-cut tabs
    ('Northwestern_TA_NLM',  drop_cat(df_ta_nlm)),
    ('Northwestern_TA_PMC',  drop_cat(df_ta_pmc)),
]

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for sheet_name, df in sheets:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        for col_cells in ws.columns:
            max_len = max(
                (len(str(cell.value)) if cell.value is not None else 0)
                for cell in col_cells
            )
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 55)

print(f"Workbook written: {output_path}")
for sheet_name, df in sheets:
    print(f"  {sheet_name:<28}: {len(df):>5,} rows")

Workbook written: output\nlm_pmc_nih_ta_journal_comparison.xlsx
  Summary                     :    16 rows
  NLM+PMC+NIH                 :   541 rows
  NLM+PMC_notNIH              :   854 rows
  NLM+NIH_notPMC              : 1,075 rows
  NLM_Only                    : 2,757 rows
  PMC+NIH_notNLM              :   407 rows
  PMC_Only                    : 2,567 rows
  NIH_Only                    :   232 rows
  Northwestern_TA_NLM         : 1,097 rows
  Northwestern_TA_PMC         :   220 rows


---
## Notes

### Northwestern TA columns on every matched row

| Column | Description |
|---|---|
| `in_northwestern_ta` | **True/False** — use this for filtering |
| `northwestern_ta_agreement` | "Wiley", "Springer Nature", or "Wiley; Springer Nature" |
| `ta_publishing_model` | Journal type from the agreement file (e.g., Hybrid, OA) |
| `ta_oa_license` | OA license from Springer Nature file (where available) |

### NIH metadata columns joined to matched rows

| Column | Description |
|---|---|
| `nih_order` | Rank by NIH publication count (1 = most NIH papers) |
| `nih_journal` | Journal title from NIH file |
| `nih_pub_count` | Papers acknowledging NIH funding, Jan–Jul 2025 |
| `nih_oa_status` | OA type (gold, hybrid, diamond, S2O) |
| `nih_publisher_type` | for profit / not for profit |
| `nih_apc_2025_usd` | 2025 APC in USD |
| `nih_apc_category` | APC band (e.g., `$6,001-$7,000`) |

### Refreshing caches
```python
pmc_df = fetch_pmc_journals(use_cache=False)
nlm_df = fetch_nlm_journals(NLM_QUERY, use_cache=False)
```
The NIH and TA files are read fresh every run (no cache) since they are local files.

### NLM query scope
| Query | Approx. scope |
|---|---|
| `currentlyindexed[All]` | Currently indexed in MEDLINE (~30k) |
| `ncbijournals[All]` | Broader NLM journal collection (~60k) |